In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
import tensorflow as tf
from keras.models import Sequential, load_model
from keras.layers import GRU, Dense, Dropout, LayerNormalization, Bidirectional, LSTM, SimpleRNN
from keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt
from keras.optimizers import Adam
from datetime import datetime



In [ ]:
df.isnull().sum()

In [ ]:
scaler = RobustScaler()
scaled_data = scaler.fit_transform(df[["y_next24h"]])

Complete Dataset Sampling

In [ ]:
from keras.preprocessing.sequence import pad_sequences
import numpy as np

X_padded = np.load("X_padded.npy", allow_pickle=True)
y_padded = np.load("y_padded.npy", allow_pickle=True)

patient_sequences = [X[~np.all(X == 0, axis=1)] for X in X_padded]
patient_labels = [y[~np.all(X == 0, axis=1)] for X, y in zip(X_padded, y_padded)]

def resample_sequences(patient_sequences, patient_labels):
    X_out, y_out = [], []

    for Xi, yi in zip(patient_sequences, patient_labels):
        X_out.append(Xi)
        seq_label = int(np.any(yi > 0))
        y_out.append(seq_label)

    X_padded = pad_sequences(X_out, padding="post", dtype="float32")

    y_out = np.array(y_out).reshape(-1, 1)

    return X_padded, y_out


X_padded, y_padded = resample_sequences(patient_sequences, patient_labels)

print("X_padded:", X_padded.shape, "| y_padded:", y_padded.shape)
print("Positives:", np.sum(y_padded), "| Negatives:", len(y_padded) - np.sum(y_padded))


X_Train & Y_Train

In [ ]:
from sklearn.model_selection import train_test_split


patient_ids = np.array([pid for pid in df['Patient_ID'].unique()])

train_ids, test_ids = train_test_split(patient_ids, test_size=0.15, random_state=42)
train_ids, val_ids  = train_test_split(train_ids, test_size=0.176, random_state=42) 
# 0.176 ≈ 15% total, so 70/15/15

pid_order = list(df['Patient_ID'].unique())
pid_to_idx = {pid:i for i,pid in enumerate(pid_order)}

train_idx = [pid_to_idx[pid] for pid in train_ids]
val_idx   = [pid_to_idx[pid] for pid in val_ids]
test_idx  = [pid_to_idx[pid] for pid in test_ids]

X_train, y_train = X_padded[train_idx], y_padded[train_idx]
X_val, y_val     = X_padded[val_idx], y_padded[val_idx]
X_test, y_test   = X_padded[test_idx], y_padded[test_idx]


# y_train = np.squeeze(y_train)
# y_val   = np.squeeze(y_val)
# y_test  = np.squeeze(y_test)

# if y_train.ndim > 1:
#     y_train = y_train[:, -1]
#     y_val   = y_val[:, -1]
#     y_test  = y_test[:, -1]

# y_train = y_train.reshape(-1, 1)
# y_val   = y_val.reshape(-1, 1)
# y_test  = y_test.reshape(-1, 1)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)



Balance Sampling [50/50 appearance of patients with incident in their admission and didnt. ]

In [ ]:
import numpy as np

balanced_X = []
balanced_y = []

pos_sequences = [(X_seq, y_seq) for X_seq, y_seq in zip(X_train, y_train) if np.any(y_seq == 1)]
neg_sequences = [(X_seq, y_seq) for X_seq, y_seq in zip(X_train, y_train) if np.all(y_seq == 0)]

print(f"Original training set: {len(X_train)} sequences")
print(f" → Positive sequences: {len(pos_sequences)}")
print(f" → Negative sequences: {len(neg_sequences)}")

for X_seq, y_seq in pos_sequences:
    balanced_X.append(X_seq)
    balanced_y.append(y_seq)

pos_count = len(pos_sequences)
if len(neg_sequences) < pos_count:
    print("Fewer negative sequences than positives. Sampling negatives with replacement.")
    sampled_neg_idx = np.random.choice(len(neg_sequences), size=pos_count, replace=True)
else:
    sampled_neg_idx = np.random.choice(len(neg_sequences), size=pos_count, replace=False)

for idx in sampled_neg_idx:
    balanced_X.append(neg_sequences[idx][0])
    balanced_y.append(neg_sequences[idx][1])

X_train = np.array(balanced_X, dtype="float32")
y_train = np.array(balanced_y, dtype="float32")

pos_after = sum(np.any(seq == 1) for seq in y_train)
neg_after = sum(np.all(seq == 0) for seq in y_train)

print("\nBalanced training set:")
print(f" → Total sequences: {len(y_train)}")
print(f" → Positive sequences: {pos_after}")
print(f" → Negative sequences: {neg_after}")
print(f" → Class ratio: {pos_after/(pos_after+neg_after):.2f} positive / {neg_after/(pos_after+neg_after):.2f} negative")


Model Training

In [ ]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Masking, LSTM, Dense, TimeDistributed
from sklearn.utils.class_weight import compute_class_weight

# shapes
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]    


model = Sequential([
    Masking(mask_value=0., input_shape=(n_timesteps, n_features)),
    Bidirectional(GRU(128, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(GRU(64, return_sequences=True)),
    Dense(1, activation="sigmoid")
])


model.compile(
    optimizer = Adam(learning_rate=1e-4, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="AUC"), tf.keras.metrics.Recall(name="recall")]
)
# model.compile(optimizer='adam', loss='mean_squared_error', metrics=[r2_score])
# model.summary()

early_stop = EarlyStopping(patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8)
y_train = y_train[..., np.newaxis]
y_val   = y_val[..., np.newaxis]
y_test  = y_test[..., np.newaxis]


y_train_flat = y_train.flatten()
mask = ~np.isnan(y_train_flat)
y_train_clean = y_train_flat[mask]

# # Compute class weights
# classes = np.unique(y_train_clean)
# class_weights = compute_class_weight(class_weight="balanced", 
#                                      classes=classes, 
#                                      y=y_train_clean.astype(int))
# cw_dict = {i: w for i, w in zip(classes, class_weights)}
# print("Class weights:", cw_dict)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=40,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_AUC", mode="max", patience=3, restore_best_weights=True
        )
    ]
)

print("Test evaluation:")
print(model.evaluate(X_test, y_test))


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"models/GRU_{timestamp}.keras"
model.save(model_filename)

In [ ]:
y_pred_probs = model.predict(X_test)
y_true = y_test.flatten()
y_pred = y_pred_probs.flatten() 


In [ ]:
# print("NaNs in y_true:", np.isnan(y_true).sum())
# print("NaNs in y_pred:", np.isnan(y_pred).sum())
# print("Shapes:", y_true.shape, y_pred.shape)


In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
)
import numpy as np
from sklearn.metrics import brier_score_loss


mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R² Score: {r2:.6f}")

threshold = 0.5
y_pred_class = (y_pred >= threshold).astype(int)

auc = roc_auc_score(y_true, y_pred)
precision = precision_score(y_true, y_pred_class, zero_division=0)
recall = recall_score(y_true, y_pred_class, zero_division=0)
f1 = f1_score(y_true, y_pred_class, zero_division=0)
accuracy = accuracy_score(y_true, y_pred_class)
brier = brier_score_loss(y_true, y_pred)

print(f"Brier Score: {brier:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("Receiver Operating Characteristic (ROC)")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.7)
plt.savefig("figures/ROCCurve_"+(timestamp)+".png")
plt.show()

precision, recall, _ = precision_recall_curve(y_true, y_pred)

plt.figure(figsize=(6,6))
plt.plot(recall, precision, color="purple", lw=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True, linestyle="--", alpha=0.7)
plt.savefig("figures/PRCurve_"+(timestamp)+".png")
plt.show()

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred_class = (y_pred >= 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title("Confusion Matrix (Threshold = 0.5)")
plt.savefig("figures/CM_"+(timestamp)+".png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score
)

thresholds = [0.3, 0.5, 0.7]

results = []

fig, axes = plt.subplots(1, len(thresholds), figsize=(15,4))

for i, thr in enumerate(thresholds):
    
    y_pred_class = (y_pred >= thr).astype(int)
    
    
    cm = confusion_matrix(y_true, y_pred_class)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues, values_format="d", ax=axes[i], colorbar=False)
    axes[i].set_title(f"Threshold = {thr}")
    
    
    precision = precision_score(y_true, y_pred_class, zero_division=0)
    recall = recall_score(y_true, y_pred_class, zero_division=0)
    f1 = f1_score(y_true, y_pred_class, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred_class)
    
    results.append({
        "Threshold": thr,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy
    })

plt.tight_layout()
plt.savefig("figures/CM_Thresholds_"+(timestamp)+".png")
plt.show()

results_df = pd.DataFrame(results)
print("\nMetrics at different thresholds:")
print(results_df.to_string(index=False, float_format="%.4f"))
